In [1]:
import mujoco
from mujoco import mjx
import jax
import jax.numpy as jnp
import scipy.optimize as optim

Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'


In [2]:
from pathlib import Path

path = Path('universal_robots_ur5e/scene.xml')
if path.exists():
    print("Path exists")
else:
    print("Path does not exist")

Path exists


In [3]:
# Load the MuJoCo model from an XML file and initialize the simulation state (CPU)
mj_model = mujoco.MjModel.from_xml_path(str(path))
mj_data = mujoco.MjData(mj_model)

# Transfer the MuJoCo model and data to MJX for hardware-accelerated simulation (GPU/TPU)
mjx_model = mjx.put_model(mj_model)
mjx_data = mjx.put_data(mj_model, mj_data)

In [4]:
# This function answers:If the joints are q, where is the robot hand?
# this code is copied from assignment

def forward_kinematics(q):

    new_mjx_data = mjx_data.replace(qpos=q)

    new_mjx_data = mjx.fwd_position(
        mjx_model,
        new_mjx_data
    )

    pos = new_mjx_data.site_xpos[
        mj_model.site('attachment_site').id
    ]

    mat = new_mjx_data.site_xmat[
        mj_model.site('attachment_site').id
    ]

    return pos, mat

In [ ]:
# Build the error fucntion
def pose_err(q, target_pos, target_mat):

    pos, mat = forward_kinematics(q)

    pos_err = jnp.sum((pos - target_pos)**2)

    rot_err = jnp.sum((mat - target_mat)**2)

    return pos_err + rot_err

In [ ]:
#jit compilation to make the run much faster
jit_err = jax.jit(pose_err)
jit_err_grad = jax.jit(jax.grad(pose_err, argnums=(0,)))
def ik_optim(target_pos, target_mat, init_guess):
    result = optim.minimize(jit_err,args=(target_pos, target_mat),x0=init_guess,
                            method='L-BFGS-B',jac=jit_err_grad)
    return result.x